# Управление потоком исполнения: if-else, граф, ReAct

Ноутбук к занятию 4 курса **«Разработка LLM-приложений с LangGraph»**.

На занятии 3 мы дошли до агента: в демо 6 написали цикл `run_with_tools`, а в бонусе увидели тот же цикл как граф в LangGraph.
Сегодня разбираем, **кто управляет потоком исполнения** — код разработчика или модель — и пробуем три подхода в одном домене договоров:

> «Прочитай все договоры в папке, найди просроченные и по каждому подготовь уведомление».

| Раздел | Что делаем | Демо на слайдах |
|---|---|---|
| 1 | Подготовка: доступ к модели и данные | — |
| 2 | if-else для разных форматов срока договора | Демо 1 |
| 3 | Цепочка вызовов с проверкой (prompt chaining) | Демо 2 |
| 4 | Повтор и исправление вручную | Демо 3 |
| 5 | Тот же процесс как граф: свой мини-движок на networkx | Демо 4 |
| 6 | Checkpointer и подтверждение человеком | Демо 5 |
| 7 | ReAct-агент для задачи про договоры | Демо 6 |
| 8 | ReAct как граф с контролем | Демо 7 |
| 9 | Итог: сравниваем подходы | — |

Весь код написан на **чистом `requests`**, как на занятии 3. Граф в разделах 5, 6 и 8 — **свой мини-движок**:
структуру хранит `networkx`, а исполнение, сохранение состояния и паузу пишем сами. LangGraph подключим на занятии 5 —
и увидим, что он делает то же самое, только надёжнее и с большим числом возможностей.


> GigaChat во Freemium обрабатывает **один запрос за раз**. Запускайте ячейки по очереди и не держите несколько вкладок с ноутбуком.
> На весь ноутбук уходит около 50 запросов к модели.

## 1. Подготовка

Ячейки этого раздела повторяют подготовку из ноутбука занятия 3: ноутбук должен работать сам по себе.

### Где взять ключ
1. Зайдите в личный кабинет GigaChat API (Studio на developers.sber.ru) через **СберID**.
2. Создайте проект **GigaChat API** и нажмите «Получить ключ».
3. Скопируйте **Authorization Key** — его показывают один раз.

### Куда положить ключ
В Colab: значок 🔑 (**Secrets**) на левой панели → **Add new secret**:
- Name: `GIGACHAT_AUTH_KEY`
- Value: ваш Authorization Key
- включите переключатель **Notebook access**.

In [ ]:
import os, json, time, uuid, re, calendar, datetime as dt
from contextlib import contextmanager
from pathlib import Path
import requests

# --- Ключ: из Colab Secrets, иначе из переменной окружения, иначе спросим ---
try:
    from google.colab import userdata
    AUTH_KEY = userdata.get("GIGACHAT_AUTH_KEY")
except Exception:
    from getpass import getpass
    AUTH_KEY = os.environ.get("GIGACHAT_AUTH_KEY") or getpass("GigaChat Authorization Key: ")
AUTH_KEY = (AUTH_KEY or "").strip()

# --- Вся конфигурация провайдера — в одной ячейке ---
SCOPE     = "GIGACHAT_API_PERS"   # физлица; для юрлиц — GIGACHAT_API_CORP / GIGACHAT_API_B2B
OAUTH_URL = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
API_URL   = "https://gigachat.devices.sberbank.ru/api/v1"
MODEL     = "GigaChat-2"          # Lite. Для агента в разделах 7–8 надёжнее "GigaChat-2-Pro"

# Тариф для учебного подсчёта стоимости, ₽ за 1000 токенов (во Freemium токены бесплатные).
PRICE_PER_1K_TOKENS = 0.0

print("Ключ загружен:", bool(AUTH_KEY))


### Сертификат и авторизация

Серверы GigaChat используют сертификат **Национального удостоверяющего центра Минцифры** — добавляем его к стандартному набору.
Затем обмениваем Authorization Key на access token, который живёт около 30 минут. Подробно — в ноутбуке занятия 3.

In [ ]:
import certifi

CERT_URL    = "https://gu-st.ru/content/lending/russian_trusted_root_ca_pem.crt"
BUNDLE_PATH = "ca_bundle_with_russian_root.pem"

try:
    if not os.path.exists(BUNDLE_PATH):
        r = requests.get(CERT_URL, timeout=30)
        r.raise_for_status()
        with open(certifi.where(), "rb") as f:
            standard = f.read()
        with open(BUNDLE_PATH, "wb") as f:
            f.write(standard + b"\n" + r.content)
    VERIFY = BUNDLE_PATH
    print("Сертификат установлен:", VERIFY)
except Exception as e:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    VERIFY = False
    print("⚠ Не удалось скачать сертификат, проверка TLS отключена:", e)


_token = {"value": None, "expires_at": 0.0}

def get_token():
    """Возвращает действующий access token, при необходимости получает новый."""
    if _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]
    resp = requests.post(
        OAUTH_URL,
        headers={"Authorization": f"Basic {AUTH_KEY}", "RqUID": str(uuid.uuid4()),
                 "Content-Type": "application/x-www-form-urlencoded", "Accept": "application/json"},
        data={"scope": SCOPE}, verify=VERIFY, timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    _token["value"] = data["access_token"]
    _token["expires_at"] = data["expires_at"] / 1000   # мс → секунды
    return _token["value"]

print("Токен получен:", get_token()[:20] + "…")

### Обёртка `chat()` и учёт токенов

`chat()` — единственное место, где мы ходим в API. `measure()` считает, сколько вызовов модели и токенов ушло на каждый подход:
в конце ноутбука сравним их между собой.

In [ ]:
USAGE_LOG = []   # usage каждого ответа модели

def chat(messages, model=None, **params):
    """POST /chat/completions. Возвращает разобранный JSON ответа целиком."""
    payload = {"model": model or MODEL, "messages": messages, **params}
    for attempt in range(3):
        resp = requests.post(
            f"{API_URL}/chat/completions",
            headers={"Authorization": f"Bearer {get_token()}",
                     "Content-Type": "application/json", "Accept": "application/json"},
            json=payload, verify=VERIFY, timeout=120,
        )
        if resp.status_code == 429:          # занят единственный поток — ждём и повторяем
            time.sleep(2 ** attempt)
            continue
        if resp.status_code == 401:          # токен протух раньше срока — сбросить и повторить
            _token["value"] = None
            continue
        if not resp.ok:
            print("Ошибка API:", resp.status_code, resp.text[:500])
        resp.raise_for_status()
        data = resp.json()
        USAGE_LOG.append(data.get("usage", {}))
        return data
    resp.raise_for_status()


def usage_report(log=None):
    log = USAGE_LOG if log is None else log
    prompt = sum(u.get("prompt_tokens", 0) for u in log)
    completion = sum(u.get("completion_tokens", 0) for u in log)
    billed = sum(u.get("total_tokens", 0) for u in log)
    print(f"Запросов: {len(log)}")
    print(f"Токены: запрос {prompt}, ответ {completion}, к оплате {billed}")
    print(f"Стоимость по тарифу {PRICE_PER_1K_TOKENS} ₽/1K: {billed / 1000 * PRICE_PER_1K_TOKENS:.2f} ₽")

In [ ]:
def reply_text(data):
    """Достаёт текст ответа из JSON."""
    return data["choices"][0]["message"]["content"]

def ask(system, user, **params):
    """Один вызов модели: системный промпт + сообщение пользователя → текст ответа."""
    return reply_text(chat([{"role": "system", "content": system},
                            {"role": "user", "content": user}], **params))

def parse_json(text):
    """Достаёт первый JSON-объект из текста (модель иногда оборачивает его в ```json ... ```)."""
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"JSON не найден в ответе: {text[:200]}")
    return json.loads(match.group(0))


STATS = []   # сводка по подходам: вызовы модели, токены, время

@contextmanager
def measure(label):
    """Считает вызовы chat() и токены внутри блока with."""
    start, t0 = len(USAGE_LOG), time.time()
    yield
    part = USAGE_LOG[start:]
    STATS.append({"подход": label, "вызовов LLM": len(part),
                  "токенов": sum(u.get("total_tokens", 0) for u in part),
                  "секунд": round(time.time() - t0, 1)})
    print(f"\n[{label}] вызовов LLM: {len(part)}, токенов: {STATS[-1]['токенов']}")

print(ask("Отвечай одним словом.", "Столица Франции?"))

### Данные для занятия

1. **Два договора с разной формулировкой срока** — для простого if-else в разделе 2:
   в первом явно указана дата окончания, во втором срок задан как «6 месяцев со дня подписания».
2. **Папка `contracts/` с пятью договорами** — для остальных разделов. Даты считаются от сегодняшнего дня, поэтому
   договоры 101 и 103 всегда просрочены, 102 и 105 действуют, а в 104 срока нет совсем — на нём будем ломать извлечение.


In [ ]:
TODAY = dt.date.today()

def day(offset):
    """Дата через offset дней от сегодня в формате ДД.ММ.ГГГГ."""
    return (TODAY + dt.timedelta(days=offset)).strftime("%d.%m.%Y")


# Два коротких документа только для демо 1. Они не попадают в папку contracts/.
TERM_DOCUMENTS = {
    "fixed-date.txt": f"""ДОГОВОР № A-01
Дата подписания: {day(-20)}.
Срок действия договора: до {day(45)}.""",
    "from-signing.txt": f"""ДОГОВОР № B-02
Дата подписания: {day(-90)}.
Срок действия договора: 6 месяцев со дня подписания.""",
}


CONTRACTS = {
    "d-101.txt": f"""ДОГОВОР ОКАЗАНИЯ УСЛУГ № 101
Заказчик: ООО «Ромашка». Исполнитель: ИП Иванов Иван Иванович.
Исполнитель обязуется оказать услуги по настройке учётной системы.
Стоимость услуг: 120 000 (сто двадцать тысяч) рублей.
Срок действия договора: до {day(-75)}.""",
    "d-102.txt": f"""ДОГОВОР ПОСТАВКИ № 102
Поставщик: АО «Север». Покупатель: ООО «Вектор».
Поставщик передаёт покупателю офисную мебель согласно спецификации.
Общая стоимость поставки: 450 000 рублей.
Договор действует до {day(200)}.""",
    "d-103.txt": f"""ДОГОВОР АРЕНДЫ № 103
Арендодатель: ИП Петрова Анна Сергеевна. Арендатор: ООО «Бета».
Предмет: нежилое помещение площадью 40 кв. м.
Арендная плата: 35 000 рублей в месяц.
Срок аренды: до {day(-12)} включительно.""",
    "d-104.txt": """ДОГОВОР ПОДРЯДА № 104
Заказчик: ООО «Гамма». Подрядчик: ООО «СтройМастер».
Подрядчик выполняет ремонт помещения по заданию заказчика.
Стоимость работ определяется сметой (Приложение 1).
Срок действия договора устанавливается дополнительным соглашением сторон.""",
    "d-105.txt": f"""ДОГОВОР ОБ ОКАЗАНИИ ПЛАТНЫХ ОБРАЗОВАТЕЛЬНЫХ УСЛУГ № 105
Исполнитель: Университет. Заказчик: Сидоров Пётр Алексеевич.
Стоимость обучения за год: сто восемьдесят тысяч рублей.
Договор действует до {day(40)}.""",
}

Path("contracts").mkdir(exist_ok=True)
for name, text in CONTRACTS.items():
    Path("contracts", name).write_text(text, encoding="utf-8")

print("Сегодня:", TODAY.strftime("%d.%m.%Y"))
print("Документы для демо 1:", list(TERM_DOCUMENTS))
for name, text in CONTRACTS.items():
    print(f"  {name}: {text.splitlines()[-1]}")


## 2. Демо 1 — if-else для срока договора

Слайды 6–7. В договорах срок может быть записан по-разному:

1. **Явная дата:** «договор действует до 15.12.2026» — используем готовый `end_date`.
2. **Относительный срок:** «6 месяцев со дня подписания» — извлекаем `signed_at` и `months`, затем считаем дату кодом.

Модель приводит текст к одной JSON-структуре и выбирает `type`. Ветки и вычисление задаёт обычный `if / elif`.


In [ ]:
TERM_SYSTEM = """Ты извлекаешь условие о сроке действия договора. Ответь ТОЛЬКО JSON:
{"type": "explicit_date" | "from_signing" | "unknown",
 "end_date": дата окончания YYYY-MM-DD или null,
 "signed_at": дата подписания YYYY-MM-DD или null,
 "months": целое число месяцев или null}

explicit_date — в тексте прямо указана дата окончания.
from_signing — срок задан количеством месяцев со дня подписания.
unknown — данных недостаточно. Ничего не придумывай."""


def add_months(date, months):
    """Прибавляет календарные месяцы, корректируя последний день месяца."""
    month_index = date.month - 1 + months
    year, month = date.year + month_index // 12, month_index % 12 + 1
    day_of_month = min(date.day, calendar.monthrange(year, month)[1])
    return date.replace(year=year, month=month, day=day_of_month)

def extract_term(text):
    return parse_json(ask(TERM_SYSTEM, text, temperature=0.1))

def normalize_end_date(term):
    """if-else выбирает один из двух заранее известных алгоритмов."""
    if term.get("type") == "explicit_date":
        if not term.get("end_date"):
            raise ValueError("модель не вернула end_date")
        return dt.date.fromisoformat(term["end_date"])
    elif term.get("type") == "from_signing":
        if not term.get("signed_at") or not isinstance(term.get("months"), int):
            raise ValueError("нужны signed_at и целое months")
        signed_at = dt.date.fromisoformat(term["signed_at"])
        return add_months(signed_at, term["months"])
    else:
        raise ValueError("срок не распознан — документ нужен человеку")


processed = []
with measure("if-else: формат срока"):
    for name, text in TERM_DOCUMENTS.items():
        term = extract_term(text)                       # LLM → одинаковый JSON
        end_date = normalize_end_date(term)             # ветку выбирает Python
        processed.append({"name": name, "type": term["type"], "end_date": end_date})
        print(f"{name}: type={term['type']}, end_date={end_date}")

processed.sort(key=lambda row: row["end_date"])         # обычная сортировка
print("\nПо возрастанию срока:")
for row in processed:
    print(f"  {row['end_date']} — {row['name']}")


Оба документа получили единое поле `end_date`, хотя срок в них сформулирован по-разному. После нормализации модель больше не нужна: список сортирует обычный Python.

Это полноценное решение для известного процесса. Даже если документов сто, внешний цикл `for` повторит те же две ветки. Само количество файлов ещё не требует агента.


In [ ]:
processed


Предел появляется, когда меняется не только формат входа, но и **сам путь обработки**. Например, для одного документа нужно запросить приложение, для другого — запустить OCR, для третьего — найти связанный договор, а после ошибки решить, какой шаг попробовать следующим.

Такие варианты тоже можно дописывать через `if-else`, но число веток растёт. Сначала рассмотрим фиксированную цепочку и ручной повтор, затем сделаем переходы явным графом и только после этого передадим выбор следующего инструмента ReAct-агенту.

**Попробуйте сами:** добавьте третий документ без срока. Он должен попасть в ветку `unknown` и потребовать человека.


## 3. Демо 2 — цепочка вызовов с проверкой

Слайд 9, паттерн **prompt chaining**. Разберём один договор так же, как в практике занятия 2, но разобьём работу на шаги:

1. **LLM** извлекает поля в JSON;
2. **код** проверяет их (gate) — если что-то не так, цепочка останавливается;
3. **LLM** оформляет уведомление, если договор просрочен.

Путь по-прежнему задан кодом: шаги идут строго друг за другом.

In [ ]:
EXTRACT_SYSTEM = """Ты извлекаешь данные из договора. Ответь ТОЛЬКО JSON без пояснений:
{"number": номер договора строкой,
 "kind": вид договора (услуги, поставка, аренда и т.п.),
 "parties": [стороны договора],
 "amount": сумма в рублях числом или null,
 "end_date": дата окончания действия в формате YYYY-MM-DD или null}
Если чего-то нет в тексте — ставь null, ничего не придумывай."""

NOTICE_SYSTEM = ("Ты юрист. Напиши короткое уведомление (2–3 предложения) второй стороне о том, "
                 "что срок действия договора истёк. Без подписи и без шаблонов в квадратных скобках.")

def read_contract(path):
    return Path(path).read_text(encoding="utf-8")

def extract_fields(text):
    return parse_json(ask(EXTRACT_SYSTEM, text, temperature=0.1))

def validate(fields):
    """Gate: обычный код, никакой модели."""
    errors = []
    if not fields.get("number"):
        errors.append("нет номера договора")
    if len(fields.get("parties") or []) < 2:
        errors.append("должно быть две стороны")
    amount = fields.get("amount")
    if not isinstance(amount, (int, float)) or amount <= 0:
        errors.append("amount должна быть положительным числом")
    try:
        dt.date.fromisoformat(str(fields.get("end_date")))
    except ValueError:
        errors.append("end_date должна быть датой YYYY-MM-DD")
    return errors

def make_notice(fields):
    return ask(NOTICE_SYSTEM, json.dumps(fields, ensure_ascii=False), temperature=0.3)


def chain(path):
    fields = extract_fields(read_contract(path))       # LLM 1: извлечь
    print("  поля:", fields)
    errors = validate(fields)                          # gate: проверить кодом
    if errors:
        return f"⛔ цепочка остановлена на проверке: {errors}"
    if dt.date.fromisoformat(fields["end_date"]) >= TODAY:
        return "договор действует — уведомление не нужно"
    return make_notice(fields)                         # LLM 2: оформить


with measure("цепочка с проверкой"):
    for path in ["contracts/d-101.txt", "contracts/d-104.txt"]:
        print(f"📄 {path}")
        print(chain(path), "\n")

Договор 104 не прошёл проверку: в нём нет ни суммы, ни срока. Цепочка честно остановилась — но сама исправить ничего не может.

**Попробуйте сами:** запустите `chain("contracts/d-105.txt")`. Сумма там написана словами — справилась ли модель с переводом в число?

## 4. Демо 3 — повтор и исправление вручную

Слайды 11 и 15. Добавим шаг «исправить»: если проверка не прошла, просим модель поправить JSON с учётом ошибок
и проверяем снова — не больше трёх раз, потом отдаём человеку.

In [ ]:
FIX_SYSTEM = """Ты исправляешь JSON с данными договора. Тебе дадут текст договора, текущий JSON и список ошибок.
Верни ТОЛЬКО исправленный JSON той же структуры. Если данных в тексте нет — оставь null, не выдумывай."""

def fix_fields(text, fields, errors):
    user = f"Текст договора:\n{text}\n\nТекущий JSON:\n{json.dumps(fields, ensure_ascii=False)}\n\nОшибки: {errors}"
    try:
        return parse_json(ask(FIX_SYSTEM, user, temperature=0.1))
    except ValueError:
        return fields                                  # модель ответила не JSON — оставляем как было


def chain_with_retry(path, max_attempts=3):
    text = read_contract(path)
    fields = extract_fields(text)
    attempts = 0
    while True:                                        # ← цикл «пока не готово»
        errors = validate(fields)
        if not errors:
            return f"✅ поля прошли проверку: {fields}"
        if attempts >= max_attempts:
            return f"🙋 нужен человек: {errors}"
        attempts += 1
        print(f"  попытка {attempts}: {errors}")
        fields = fix_fields(text, fields, errors)


with measure("цепочка с повтором"):
    print(chain_with_retry("contracts/d-104.txt"))

Это уже **граф**, только нарисованный неявно: узлы «извлечь», «проверить», «исправить», ребро назад и условие выхода по счётчику.
Пока код маленький, это не страшно. Но как только понадобится остановиться и подождать человека, продолжить после сбоя
или посмотреть, на каком шаге мы сейчас, всё это придётся писать самим. Дальше то же самое опишем явно.

## 5. Демо 4 — тот же процесс как граф

Слайды 14, 16 и 17. Пять понятий:

- **состояние** — словарь, общий для всех узлов;
- **узел** — функция «состояние → обновление состояния»;
- **ребро** — безусловный переход;
- **условное ребро** — функция `route()` выбирает следующий узел;
- **цикл** — ребро `fix → validate`.

LangGraph подключим на занятии 5. Чтобы было понятно, что он делает внутри, сначала напишем **свой мини-движок графа**
с тем же интерфейсом: `add_node`, `add_edge`, `add_conditional_edges`, `START`, `END`.

Структуру графа храним в **networkx** (`nx.DiGraph`): он же проверяет, что все узлы достижимы, ищет циклы и рисует схему.
Наш код отвечает только за исполнение: какой узел запустить следующим, как сохранить состояние и как встать на паузу.
На занятии 5 заменим `MiniGraph` на LangGraph почти без изменений в остальном коде.

In [ ]:
import copy
import networkx as nx
import matplotlib.pyplot as plt

START, END = "__start__", "__end__"


class GraphInterrupt(Exception):
    """Узел попросил паузу: граф сохраняет состояние и ждёт решения человека."""
    def __init__(self, value):
        super().__init__(value)
        self.value = value

_resume = []   # ответ человека для узла, который продолжаем после паузы

def interrupt(value):
    """Пауза внутри узла. При продолжении узел запускается заново, и interrupt() возвращает ответ человека."""
    if _resume:
        return _resume.pop()
    raise GraphInterrupt(value)


class MiniGraph:
    """Учебный граф с тем же интерфейсом, что StateGraph в LangGraph.
    Структуру хранит networkx.DiGraph, исполнение по шагам и checkpointer — наш код."""

    def __init__(self):
        self.g = nx.DiGraph()
        self.g.add_nodes_from([START, END])
        self.routers = {}              # узел → (функция-маршрутизатор, {метка: следующий узел})
        self.checkpoints = {}          # thread_id → список снимков состояния: наш checkpointer
        self.last_state = None

    # --- описание графа ---
    def add_node(self, name, fn):
        self.g.add_node(name, fn=fn)

    def add_edge(self, src, dst):
        self.g.add_edge(src, dst, label="")

    def add_conditional_edges(self, src, router, mapping):
        self.routers[src] = (router, mapping)
        for label, dst in mapping.items():
            self.g.add_edge(src, dst, label=label, conditional=True)

    def compile(self):
        """Проверки графа силами networkx."""
        no_fn = [n for n in self.g if n not in (START, END) and "fn" not in self.g.nodes[n]]
        if no_fn:
            raise ValueError(f"В рёбрах есть узлы без функции: {no_fn}")
        unreachable = set(self.g) - nx.descendants(self.g, START) - {START}
        if unreachable:
            raise ValueError(f"Узлы недостижимы из START: {unreachable}")
        if not nx.has_path(self.g, START, END):
            raise ValueError("Из START нельзя дойти до END")
        return self

    # --- исполнение ---
    def _next(self, node, state):
        if node in self.routers:
            router, mapping = self.routers[node]
            return mapping[router(state)]
        successors = list(self.g.successors(node))
        return successors[0] if successors else END

    def _save(self, thread_id, state, next_node, interrupt_value=None):
        if thread_id is None:
            return
        history = self.checkpoints.setdefault(thread_id, [])
        history.append({"step": len(history), "next": next_node,
                        "state": copy.deepcopy(state), "interrupt": interrupt_value})

    def stream(self, inputs=None, thread_id=None, resume=None, max_steps=50):
        """Выполняет граф и отдаёт (узел, обновление) после каждого узла."""
        if resume is not None:                         # продолжаем с места паузы
            last = self.get_state(thread_id)
            if not last or last["interrupt"] is None:
                raise ValueError("Продолжать нечего: граф не стоит на паузе")
            state, node = copy.deepcopy(last["state"]), last["next"]
        else:
            state = dict(inputs or {})
            node = self._next(START, state)

        for _ in range(max_steps):
            if node == END:
                return
            _resume[:] = [resume] if resume is not None else []
            resume = None                              # ответ человека — только для первого узла
            try:
                update = self.g.nodes[node]["fn"](state) or {}
            except GraphInterrupt as e:
                if thread_id is None:
                    raise RuntimeError("Для паузы нужен thread_id: без checkpointer состояние негде сохранить")
                self._save(thread_id, state, node, e.value)
                yield "__interrupt__", e.value
                return
            finally:
                _resume.clear()
            state.update(update)
            self.last_state = state
            done, node = node, self._next(node, state)
            self._save(thread_id, state, node)
            yield done, update
        raise RuntimeError(f"Превышен лимит шагов графа: {max_steps}")

    def invoke(self, inputs=None, thread_id=None, resume=None, max_steps=50):
        for _ in self.stream(inputs, thread_id, resume, max_steps):
            pass
        return self.last_state

    # --- то, что даёт checkpointer ---
    def get_state(self, thread_id):
        history = self.checkpoints.get(thread_id)
        return history[-1] if history else None

    def get_state_history(self, thread_id):
        return list(reversed(self.checkpoints.get(thread_id, [])))


def draw_graph(app, llm_nodes=()):
    """Рисует граф: оранжевые узлы вызывают LLM, голубые — обычный код; условные рёбра пунктиром."""
    G = app.g
    depth = nx.single_source_shortest_path_length(G, START)
    for n in G:
        G.nodes[n]["layer"] = depth.get(n, max(depth.values()) + 1)
    G.nodes[END]["layer"] = max(G.nodes[p]["layer"] for p in G.predecessors(END)) + 1   # END — после всех, кто в него ведёт
    pos = nx.multipartite_layout(G, subset_key="layer")

    colors = ["#5A6474" if n in (START, END) else "#F3C9AE" if n in llm_nodes else "#C9D3DD" for n in G]
    names = {n: "START" if n == START else "END" if n == END else n for n in G}
    plain = [(u, v) for u, v, d in G.edges(data=True) if not d.get("conditional")]
    cond = [(u, v) for u, v, d in G.edges(data=True) if d.get("conditional")]
    labels = {(u, v): "END" if d["label"] == END else d["label"] for u, v, d in G.edges(data=True) if d.get("label")}
    arc = "arc3,rad=0.15"

    plt.figure(figsize=(10, 4.5))
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=2600)
    nx.draw_networkx_labels(G, pos, labels=names, font_size=9)
    nx.draw_networkx_edges(G, pos, edgelist=plain, node_size=2600, arrowsize=16, connectionstyle=arc)
    nx.draw_networkx_edges(G, pos, edgelist=cond, node_size=2600, arrowsize=16, connectionstyle=arc,
                           style="dashed", edge_color="#B4501E")
    try:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=8, connectionstyle=arc)
    except TypeError:                                  # старые версии networkx
        nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=8)
    plt.axis("off")
    plt.show()

    cycles = [c for c in nx.simple_cycles(G)]
    print("Циклы:", cycles or "нет — это конвейер (DAG)")

Теперь описываем процесс из разделов 3–4 как граф. Внутри узлов — **те же функции** на `requests`:
движку графа всё равно, как узел обращается к модели. Сравните с кодом на слайде 17 — отличается только класс графа.

In [ ]:
MAX_ATTEMPTS = 3
Path("extracted").mkdir(exist_ok=True)

def extract(state):                                    # LLM
    text = read_contract(state["path"])
    return {"text": text, "fields": extract_fields(text), "attempts": 0}

def validate_node(state):                              # код
    return {"errors": validate(state["fields"])}

def fix(state):                                        # LLM
    return {"fields": fix_fields(state["text"], state["fields"], state["errors"]),
            "attempts": state["attempts"] + 1}

def save(state):                                       # код
    out = Path("extracted", Path(state["path"]).stem + ".json")
    out.write_text(json.dumps(state["fields"], ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status": f"сохранено в {out}"}

def to_human(state):                                   # код
    return {"status": f"нужен человек: {state['errors']}"}

def route(state):                                      # условное ребро: решает код
    if not state["errors"]:
        return "ok"
    if state["attempts"] < MAX_ATTEMPTS:
        return "retry"
    return "human"


g = MiniGraph()
g.add_node("extract", extract)
g.add_node("validate", validate_node)
g.add_node("fix", fix)
g.add_node("save", save)
g.add_node("human", to_human)

g.add_edge(START, "extract")
g.add_edge("extract", "validate")
g.add_conditional_edges("validate", route, {"ok": "save", "retry": "fix", "human": "human"})
g.add_edge("fix", "validate")
g.add_edge("save", END)
g.add_edge("human", END)

doc_app = g.compile()
draw_graph(doc_app, llm_nodes={"extract", "fix"})   # сверяем со схемой на слайде 16

Запускаем через `stream`: граф отдаёт обновление после каждого узла, и мы видим путь исполнения.

In [ ]:
def short(update):
    """Краткое описание обновления состояния: без полного текста договора."""
    shown = {k: v for k, v in (update or {}).items() if k != "text"}
    return json.dumps(shown, ensure_ascii=False, default=str)[:150]

def run_graph(app, inputs=None, thread_id=None, resume=None):
    for node, update in app.stream(inputs, thread_id=thread_id, resume=resume):
        if node == "__interrupt__":
            print("  ⏸ граф остановился и ждёт человека")
        else:
            print(f"  → {node}: {short(update)}")


with measure("граф: обработка договора"):
    for path in ["contracts/d-101.txt", "contracts/d-104.txt"]:
        print(f"📄 {path}")
        run_graph(doc_app, {"path": path})
        print()

Поведение то же, что в разделе 4, но поток теперь описан явно: у каждого узла понятный вход и выход,
а `route()` легко протестировать отдельно — это обычная функция от словаря.

**Попробуйте сами:** поменяйте `MAX_ATTEMPTS` на 1 и посмотрите, как изменился путь для договора 104.

## 6. Демо 5 — checkpointer и подтверждение человеком

Слайд 19: что граф даёт сверх обычного цикла. В нашем мини-движке это уже есть:

- **checkpointer** — после каждого шага `MiniGraph` кладёт снимок состояния в `checkpoints[thread_id]`.
  По этому снимку граф можно продолжить и посмотреть историю шагов;
- **`interrupt()`** — узел бросает `GraphInterrupt`, движок сохраняет состояние и останавливается.
  Продолжаем через `stream(..., resume=ответ)`: узел запускается заново, и `interrupt()` возвращает ответ человека.

В LangGraph это `MemorySaver` (или checkpointer на базе данных), `interrupt()` и `Command(resume=...)` — посмотрим на занятии 5.

Добавим перед сохранением узел `approve`: человек видит извлечённые поля и решает, сохранять ли их.

In [ ]:
def approve(state):
    """Пауза: граф останавливается и ждёт решения человека."""
    decision = interrupt({"вопрос": "Сохранить извлечённые поля?", "поля": state["fields"]})
    return {"status": "одобрено" if decision == "да" else "отклонено человеком"}

def after_approve(state):
    return "save" if state["status"] == "одобрено" else "end"


g2 = MiniGraph()
for name, fn in [("extract", extract), ("validate", validate_node), ("fix", fix),
                 ("approve", approve), ("save", save), ("human", to_human)]:
    g2.add_node(name, fn)

g2.add_edge(START, "extract")
g2.add_edge("extract", "validate")
g2.add_conditional_edges("validate", route, {"ok": "approve", "retry": "fix", "human": "human"})
g2.add_edge("fix", "validate")
g2.add_conditional_edges("approve", after_approve, {"save": "save", "end": END})
g2.add_edge("save", END)
g2.add_edge("human", END)

doc_app2 = g2.compile()
draw_graph(doc_app2, llm_nodes={"extract", "fix"})

In [ ]:
# thread_id — «номер разговора»: по нему checkpointer находит сохранённое состояние
THREAD = "contract-102"

print("📄 contracts/d-102.txt")
run_graph(doc_app2, {"path": "contracts/d-102.txt"}, thread_id=THREAD)

snapshot = doc_app2.get_state(THREAD)
print("\nСледующий узел:", snapshot["next"])
print("Граф спрашивает:", snapshot["interrupt"])

Граф стоит на паузе, а его состояние лежит в checkpointer. Модель больше не вызывается: продолжим с того же места,
даже если между паузой и ответом пройдёт час (в нашем движке состояние живёт в памяти; в продакшене его хранят в базе данных).

Отвечаем «да» или «нет». Чтобы ячейка не ждала ввода при запуске всего ноутбука, задайте `AUTO_DECISION = "да"`.

In [ ]:
AUTO_DECISION = None   # None — спросить через input(); "да" / "нет" — ответить автоматически

decision = AUTO_DECISION or input("Сохранить поля? (да/нет): ").strip().lower()
run_graph(doc_app2, thread_id=THREAD, resume=decision)
print("Статус:", doc_app2.get_state(THREAD)["state"].get("status"))

In [ ]:
# История шагов из checkpointer: от последнего к первому
for snap in doc_app2.get_state_history(THREAD):
    pause = "  ⏸ пауза" if snap["interrupt"] else ""
    print(f"шаг {snap['step']}: следующий узел {snap['next']}, статус {snap['state'].get('status')}{pause}")

Связь с занятием 3: checkpointer — это «память» без ручного хранения `messages`, а подтверждение человеком — последний рубеж
защиты, когда инъекция может превратиться в действие.

**Попробуйте сами:** запустите тот же граф для `d-101.txt` с другим `thread_id` и ответьте «нет». Какой путь прошёл граф?

## 7. Демо 6 — ReAct-агент для задачи про договоры

Слайды 23–26. Граф хорош, когда путь известен. В нашей задаче он неизвестен: сколько договоров, какие просрочены, сколько уведомлений —
выясняется по ходу. Отдадим выбор следующего шага модели.

Инструменты агента — обычные функции:
`list_files`, `read_document`, `get_current_date`, `days_between` и `save_note`.

In [ ]:
def list_files(folder="contracts"):
    p = Path(folder)
    if not p.is_dir():
        raise FileNotFoundError(f"папка {folder} не найдена")
    return {"files": sorted(str(f) for f in p.iterdir() if f.is_file())}

def read_document(path):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"файл {path} не найден")
    return {"path": str(p), "text": p.read_text(encoding="utf-8")}

def get_current_date():
    return {"date": dt.date.today().isoformat()}

def days_between(date_from, date_to):
    return {"days": (dt.date.fromisoformat(date_to) - dt.date.fromisoformat(date_from)).days}

def save_note(name, text):
    safe = re.sub(r"[^\w\-]", "_", name)[:60]
    out = Path("notes", f"{safe}.txt")
    out.write_text(text, encoding="utf-8")
    return {"saved": str(out)}

AGENT_TOOLS = {"list_files": list_files, "read_document": read_document, "get_current_date": get_current_date,
               "days_between": days_between, "save_note": save_note}

FUNCTIONS = [
    {"name": "list_files", "description": "Возвращает список файлов в папке.",
     "parameters": {"type": "object",
                    "properties": {"folder": {"type": "string", "description": "Папка, например contracts"}},
                    "required": ["folder"]}},
    {"name": "read_document", "description": "Читает текст договора по пути к файлу.",
     "parameters": {"type": "object",
                    "properties": {"path": {"type": "string", "description": "Путь, например contracts/d-101.txt"}},
                    "required": ["path"]}},
    {"name": "get_current_date", "description": "Возвращает сегодняшнюю дату в формате YYYY-MM-DD.",
     "parameters": {"type": "object", "properties": {}}},
    {"name": "days_between", "description": "Считает количество дней между двумя датами. Отрицательное число — вторая дата раньше первой.",
     "parameters": {"type": "object",
                    "properties": {"date_from": {"type": "string", "description": "Дата начала, YYYY-MM-DD"},
                                   "date_to": {"type": "string", "description": "Дата конца, YYYY-MM-DD"}},
                    "required": ["date_from", "date_to"]}},
    {"name": "save_note", "description": "Сохраняет текст уведомления в файл.",
     "parameters": {"type": "object",
                    "properties": {"name": {"type": "string", "description": "Имя уведомления, например notice-101"},
                                   "text": {"type": "string", "description": "Текст уведомления"}},
                    "required": ["name", "text"]}},
]

AGENT_SYSTEM = ("Ты помощник юриста. Работай только через функции: смотри список файлов, читай договоры, "
                "узнавай сегодняшнюю дату. Даты в договорах записаны как ДД.ММ.ГГГГ. "
                "Уведомления сохраняй функцией save_note. Когда всё сделано — кратко перечисли, что сделано.")

### Цикл агента

Это `run_with_tools` из демо 6 занятия 3 с тремя доработками со слайда 26:
печатаем траекторию в терминах ReAct, сохраняем все вызовы в `TRACE` и останавливаемся, если модель повторяет тот же вызов.

Напоминание о формате GigaChat: функции передаём в поле `functions`, вызов приходит в `message.function_call`
с `finish_reason = "function_call"`, результат возвращаем сообщением с ролью `function`.

In [ ]:
TRACE = []   # все вызовы инструментов последнего запуска

def run_agent(task, max_steps=15, verbose=True):
    TRACE.clear()
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": task}]

    for step in range(1, max_steps + 1):
        data = chat(messages, functions=FUNCTIONS, function_call="auto")
        choice = data["choices"][0]
        msg = choice["message"]
        messages.append(msg)                                  # ответ модели — в историю целиком

        if choice.get("finish_reason") == "error":            # GigaChat: некорректные аргументы
            return f"⚠ Модель вернула некорректный вызов функции: {msg.get('content')}"
        if choice.get("finish_reason") != "function_call":
            return msg["content"]                             # Final Answer: модель решила, что готово

        if verbose and msg.get("content"):
            print(f"Thought: {msg['content']}")
        call = msg["function_call"]
        name, args = call["name"], call.get("arguments") or {}
        if isinstance(args, str):
            args = json.loads(args)

        if TRACE and TRACE[-1] == (name, args):               # зацикливание: тот же вызов подряд
            return f"⚠ Остановились: модель повторяет вызов {name}"
        TRACE.append((name, args))

        try:
            result = AGENT_TOOLS[name](**args)                # Action выполняет наш код
        except Exception as e:
            result = {"error": str(e)}                        # ошибка — тоже наблюдение
        if verbose:
            print(f"Action: {name}({json.dumps(args, ensure_ascii=False)})")
            print(f"Observation: {json.dumps(result, ensure_ascii=False)[:150]}")

        messages.append({"role": "function", "name": name,
                         "content": json.dumps(result, ensure_ascii=False)})

    return "⚠ Превышен лимит шагов"

In [ ]:
TASK = ("В папке contracts лежат договоры. Найди те, у которых срок действия уже истёк на сегодня, "
        "и по каждому сохрани короткое уведомление через save_note с именем notice-<номер договора>. "
        "В конце перечисли, что сделано.")

notes = Path("notes")
notes.mkdir(exist_ok=True)
for f in notes.glob("*.txt"):
    f.unlink()                                       # начинаем с пустой папки

start = len(USAGE_LOG)
with measure("ReAct: агент по договорам"):
    answer = run_agent(TASK)
print("\nFinal Answer:", answer)

In [ ]:
reads = [args for name, args in TRACE if name == "read_document"]
print(f"read_document вызван {len(reads)} раз — этого числа нет нигде в коде, его определил list_files")
print("Сохранённые уведомления:", sorted(p.name for p in Path("notes").glob("*.txt")))

# Рост контекста: каждый шаг отправляет всю историю заново
print("Токенов в запросе по шагам:", [u.get("prompt_tokens") for u in USAGE_LOG[start:]])

Обратите внимание на последнюю строку: каждый шаг агента дороже предыдущего, потому что история растёт. Это та же «цена памяти»,
что на занятии 3.

### Инженерные вопросы цикла

Три проверки со слайда 26. **Ошибка инструмента** — модель получает её как наблюдение и может отреагировать:

In [ ]:
print(run_agent("Прочитай договор contracts/d-999.txt и назови его сумму."))

**Лимит шагов** — защита от бесконечного цикла и перерасхода. С `max_steps=2` агент не успеет закончить работу:

In [ ]:
print(run_agent(TASK, max_steps=2))

**Зацикливание** — если модель дважды подряд делает тот же вызов, `run_agent` останавливается (проверка `TRACE[-1]`).
Вызвать это специально трудно: обычно случается на слабых моделях и длинных задачах.

**Попробуйте сами:** уберите из `AGENT_SYSTEM` фразу про формат дат и запустите задачу ещё раз. Ошибается ли модель с просрочкой?

## 8. Демо 7 — ReAct как граф с контролем

Слайды 29 и 33. Цикл из раздела 7 — это граф из двух узлов: `agent` вызывает модель, `tools` выполняет инструменты,
а условное ребро `tools_condition` решает, идти ли дальше. Мы видели его в бонусе занятия 3, а сейчас соберём на своём `MiniGraph`.

Раз это граф, в него можно встроить контроль, которого нет в голом цикле. Сделаем так, чтобы **`save_note` не выполнялся без
подтверждения человека**: узел `tools` вызывает `interrupt()` перед опасным действием.

Узел `tools` выполняет **по одному вызову за шаг**: если модель попросила несколько вызовов сразу, он вернётся сам в себя.
Так пауза и продолжение никогда не повторят уже выполненные действия.

In [ ]:
def call_model(messages):
    """Один шаг модели с описаниями функций — ответ модели целиком."""
    return chat(messages, functions=FUNCTIONS, function_call="auto")["choices"][0]["message"]

def pending_calls(messages):
    """Вызовы функций из последнего ответа модели, на которые ещё нет результата."""
    last = messages[-1]
    if last.get("role") == "assistant" and last.get("function_call"):
        call = last["function_call"]
        args = call.get("arguments") or {}
        if isinstance(args, str):
            args = json.loads(args)
        return [{"id": None, "name": call["name"], "args": args}]
    return []

def tool_message(call, result):
    return {"role": "function", "name": call["name"], "content": json.dumps(result, ensure_ascii=False)}

In [ ]:
APPROVAL_TOOLS = {"save_note"}   # действия, которые выполняются только после подтверждения человека

def agent_node(state):                                 # LLM
    return {"messages": state["messages"] + [call_model(state["messages"])]}

def tools_node(state):                                 # код
    call = pending_calls(state["messages"])[0]         # по одному вызову за шаг
    if call["name"] in APPROVAL_TOOLS:
        decision = interrupt({"действие": call["name"], "args": call["args"]})   # ← ждём человека
        if decision != "да":
            result = {"error": "человек отклонил действие"}
            return {"messages": state["messages"] + [tool_message(call, result)]}
    try:
        result = AGENT_TOOLS[call["name"]](**call["args"])
    except Exception as e:
        result = {"error": str(e)}
    return {"messages": state["messages"] + [tool_message(call, result)]}

def tools_condition(state):
    """После модели: есть вызовы — в tools, нет — конец."""
    return "tools" if pending_calls(state["messages"]) else END

def after_tools(state):
    """После инструмента: остались вызовы — ещё раз tools, иначе снова к модели."""
    return "tools" if pending_calls(state["messages"]) else "agent"


g3 = MiniGraph()
g3.add_node("agent", agent_node)
g3.add_node("tools", tools_node)
g3.add_edge(START, "agent")
g3.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
g3.add_conditional_edges("tools", after_tools, {"tools": "tools", "agent": "agent"})

react_app = g3.compile()
draw_graph(react_app, llm_nodes={"agent"})

`max_steps` у графа — тот же лимит шагов, что в разделе 7: при превышении движок останавливается с ошибкой.

In [ ]:
def run_react(task, thread_id, max_steps=40):
    inputs = {"messages": [{"role": "system", "content": AGENT_SYSTEM},
                           {"role": "user", "content": task}]}
    resume = None
    while True:
        for node, update in react_app.stream(None if resume else inputs, thread_id=thread_id,
                                             resume=resume, max_steps=max_steps):
            if node == "__interrupt__":
                continue
            msg = update["messages"][-1]
            if node == "agent":
                calls = pending_calls(update["messages"])
                if calls and msg.get("content"):
                    print(f"Thought: {msg['content']}")
                for c in calls:
                    print(f"Action: {c['name']}({json.dumps(c['args'], ensure_ascii=False)})")
                if not calls:
                    print(f"\nFinal Answer: {msg['content']}")
            else:
                print(f"Observation: {msg['content'][:150]}")

        snapshot = react_app.get_state(thread_id)
        if snapshot["next"] == END:                    # граф дошёл до конца
            return
        request = snapshot["interrupt"]
        print(f"\n⏸ Агент хочет выполнить {request['действие']}:\n"
              f"{json.dumps(request['args'], ensure_ascii=False, indent=2)}\n")
        resume = AUTO_DECISION or input("Разрешить? (да/нет): ").strip().lower()   # продолжаем с того же места


for f in Path("notes").glob("*.txt"):
    f.unlink()
with measure("ReAct-граф с подтверждением"):
    run_react(TASK, thread_id="react-1")
print("\nСохранённые уведомления:", sorted(p.name for p in Path("notes").glob("*.txt")))

Модель по-прежнему сама выбирает путь, но опасное действие проходит через человека. Так выглядит типичная продакшен-архитектура
со слайда 33: детерминированный каркас и автономия внутри одного узла.

**Попробуйте сами:** на первом запросе ответьте «нет». Что агент сделает дальше — попробует снова, пропустит договор или сообщит об отказе?

## 9. Итог: сравниваем подходы

Сколько вызовов модели и токенов ушло на каждый подход в этом запуске:

In [ ]:
import pandas as pd
pd.DataFrame(STATS)

| | if-else | Граф | ReAct |
|---|---|---|---|
| Кто выбирает следующий шаг | код | код + LLM на условных рёбрах | LLM |
| Число шагов | фиксировано | ограничено топологией | динамическое |
| Предсказуемость и тестируемость | высокая | средняя или высокая | низкая |
| Стоимость | минимальная | предсказуемая | плавающая |
| Наш пример | нормализация срока договора | обработка одного договора | все договоры в папке |

Главное правило: **минимальная автономия, достаточная для задачи**. Простого if-else достаточно для двух известных форматов срока,
а агент для одного договора был бы лишним — граф там проще, дешевле и предсказуемее.

**На занятии 5** заменим `MiniGraph` на LangGraph: те же узлы и рёбра, плюс готовые checkpointer на базе данных,
`interrupt()`, стриминг и инструменты отладки.

**Домашнее задание:** нарисуйте поток управления своей курсовой — узлы, переходы и места, где решает модель.
Какие действия в ней должны проходить через подтверждение человека?


In [ ]:
usage_report()   # сколько токенов ушло за весь ноутбук